# Tokenize Raw Data

Tokenizes raw text data with any tokenizer.
Run this for each tokenizer you want to compare.

In [ ]:
from pathlib import Path
from datasets import load_from_disk, Dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

In [ ]:
# ===================
# CONFIGURATION
# ===================

# Path to raw text data (from notebook 01)
RAW_DATA_DIR = Path("../data/fineweb-edu-raw")

# Tokenizer - change this for each tokenizer you want to test
TOKENIZER_PATH = "gpt2"  # or path to your custom tokenizer

# Output directory - will be named after tokenizer
TOKENIZER_NAME = Path(TOKENIZER_PATH).name if "/" in TOKENIZER_PATH else TOKENIZER_PATH
OUTPUT_DIR = Path(f"../data/fineweb-edu-{TOKENIZER_NAME}")

# Number of processes for parallel tokenization
NUM_PROC = 8

## 1. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Vocab size: {len(tokenizer):,}")
print(f"EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"Output will be saved to: {OUTPUT_DIR}")

## 2. Load Raw Data

In [ ]:
train_raw = load_from_disk(str(RAW_DATA_DIR / "train"))
val_raw = load_from_disk(str(RAW_DATA_DIR / "val"))
test_raw = load_from_disk(str(RAW_DATA_DIR / "test"))

print(f"Train: {len(train_raw):,} documents")
print(f"Val:   {len(val_raw):,} documents")
print(f"Test:  {len(test_raw):,} documents")

## 3. Tokenize

In [ ]:
def tokenize_fn(examples):
    """Tokenize text, keeping uid."""
    tokens = tokenizer(
        examples["text"],
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
    )
    return {
        "input_ids": tokens["input_ids"],
        "uid": examples["uid"],
    }

In [ ]:
print("Tokenizing train set...")
train_tok = train_raw.map(
    tokenize_fn,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=["text"],
    desc="Tokenizing train",
)

print("Tokenizing val set...")
val_tok = val_raw.map(
    tokenize_fn,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=["text"],
    desc="Tokenizing val",
)

print("Tokenizing test set...")
test_tok = test_raw.map(
    tokenize_fn,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=["text"],
    desc="Tokenizing test",
)

## 4. Compute Statistics

In [ ]:
train_tokens = sum(len(x) for x in train_tok["input_ids"])
val_tokens = sum(len(x) for x in val_tok["input_ids"])
test_tokens = sum(len(x) for x in test_tok["input_ids"])

print(f"\nTokenizer: {TOKENIZER_NAME}")
print(f"Vocab size: {len(tokenizer):,}")
print(f"")
print(f"Train: {train_tokens:,} tokens ({train_tokens/1e9:.2f}B)")
print(f"Val:   {val_tokens:,} tokens ({val_tokens/1e6:.0f}M)")
print(f"Test:  {test_tokens:,} tokens ({test_tokens/1e6:.0f}M)")
print(f"Total: {(train_tokens+val_tokens+test_tokens):,} tokens")

## 5. Save Tokenized Data

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_tok.save_to_disk(OUTPUT_DIR / "train")
val_tok.save_to_disk(OUTPUT_DIR / "val")
test_tok.save_to_disk(OUTPUT_DIR / "test")

print(f"Saved to {OUTPUT_DIR}")

## 6. Verify

In [ ]:
check = load_from_disk(str(OUTPUT_DIR / "train"))
print(f"Columns: {check.column_names}")
print(f"First doc length: {len(check[0]['input_ids'])} tokens")
print(f"First 20 tokens: {check[0]['input_ids'][:20]}")
print(f"Decoded: {tokenizer.decode(check[0]['input_ids'][:50])}...")

## Done!

Update `config.yaml`:

```yaml
paths:
  tokenizer: <your-tokenizer-path>
  train_data: ./data/fineweb-edu-<tokenizer-name>/train
  val_data: ./data/fineweb-edu-<tokenizer-name>/val
  test_data: ./data/fineweb-edu-<tokenizer-name>/test
```

To tokenize with another tokenizer, change `TOKENIZER_PATH` and run again.